In [ ]:
# ライブラリインポート
import numpy as np
import pandas as pd
import marimo as mo

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from IPython.display import display

from xgboost import XGBRegressor

1. ライブラリのインポートとデータの読み込み

まずは分析に必要なライブラリを読み込み、学習データ・テストデータ・提出用サンプルファイルを読み込む。
その後、データの先頭を確認して全体像を把握する。

In [ ]:
# データの読み込み
train = pd.read_csv("C:/Users/masas/kaggle_competitions/kaggle_student_test_scores_prediction/data/train.csv")
test = pd.read_csv("C:/Users/masas/kaggle_competitions/kaggle_student_test_scores_prediction/data/test.csv")
sample_submission = pd.read_csv("C:/Users/masas/kaggle_competitions/kaggle_student_test_scores_prediction/data/sample_submission.csv")

In [ ]:
# 読み込んだデータの先頭を確認
display("## Train Data")
display(train.head())

display("## Test Data")
display(test.head())

## Train Data id age gender course study_hours class_attendance internet_access sleep_hours sleep_quality study_method facility_rating exam_difficulty exam_score 0 0 21 female b.sc 7.91 98.8 no 4.9 average online videos low easy 78.3 1 1 18 other diploma 4.95 94.8 yes 4.7 poor self-study medium moderate 46.7 2 2 20 female b.sc 4.68 92.6 yes 5.8 poor coaching high moderate 99.0 3 3 19 male b.sc 2.00 49.5 yes 8.3 average group study high moderate 63.9 4 4 23 male bca 7.65 86.9 yes 9.6 good self-study high easy 100.0 ## Test Data id age gender course study_hours class_attendance internet_access sleep_hours sleep_quality study_method facility_rating exam_difficulty 0 630000 24 other ba 6.85 65.2 yes 5.2 poor group study high easy 1 630001 18 male diploma 6.61 45.0 no 9.3 poor coaching low easy 2 630002 24 female b.tech 6.60 98.5 yes 6.2 good group study medium moderate 3 630003 24 male diploma 3.03 66.3 yes 5.7 average mixed medium moderate 4 630004 20 female b.tech 2.03 42.4 yes 9.2 average coaching low moderate

2. 説明変数と目的変数の準備

学習データから目的変数 exam_score を取り出し、モデルに入力する説明変数を作成する。
また、数値列とカテゴリ列を明示的に分けておく。

In [ ]:
# 目的変数と説明変数を分離し、"id"を削除
X = train.drop(columns=["id", "exam_score"])
y = train["exam_score"]

# テストデータも同じく "id" を削除
X_test = test.drop(columns=["id"])

# 数値変数のカラムを指定
numeric_cols = [
    "age",
    "study_hours",
    "class_attendance",
    "sleep_hours",
]

# カテゴリ変数のカラムを指定
categorical_cols = [
    "gender",
    "course",
    "internet_access",
    "sleep_quality",
    "study_method",
    "facility_rating",
    "exam_difficulty",
]

3. 前処理の設定

数値変数はそのまま使用し、カテゴリ変数についてはワンホットエンコーディングを行う。
ColumnTransformer を使うことで、列ごとに異なる前処理をまとめて管理できる。

In [ ]:
# 前処理の定義
# 数値変数はそのまま使用し、カテゴリ変数はOneHotEncoderで数値化する
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ]
)

4. モデルの構築

今回は回帰モデルとして XGBRegressor を使用する。
ベースラインのため、木の本数や学習率などの基本的なハイパーパラメータを設定している。

In [ ]:
# XGBoost回帰モデルの定義
model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

5. 交差検証の準備

モデルの汎化性能を確認するために、5分割のKFold交差検証を行う。
OOF予測とテストデータ予測を保存する配列もここで用意する。

6. 学習と検証

各foldごとに学習データと検証データに分け、
前処理とモデルをまとめた Pipeline で学習・予測を行う。
各foldのRMSEを確認しながら、最終的なOOF予測も作成する。

In [ ]:
# 5-foldの交差検証を設定
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# OOF予測とテスト予測を保存する配列を用意
oof_pred = np.zeros(len(X))
test_pred = np.zeros(len(X_test))


# RMSEを計算する関数
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y), start=1):
    # foldごとに学習用データと検証用データを分割
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    # 前処理 + モデルをまとめたパイプラインを作成
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])

    # 学習
    pipeline.fit(X_train, y_train)

    # 検証データに対する予測
    valid_pred = pipeline.predict(X_valid)
    oof_pred[valid_idx] = valid_pred

    # テストデータに対する予測をfoldごとに平均化
    fold_test_pred = pipeline.predict(X_test)
    test_pred += fold_test_pred / kf.n_splits

    # foldごとのRMSEを計算
    fold_rmse = rmse(y_valid, valid_pred)
    print(f"Fold {fold}: RMSE = {fold_rmse:.5f}")

Fold 1: RMSE = 8.74287
Fold 2: RMSE = 8.74805
Fold 3: RMSE = 8.74241
Fold 4: RMSE = 8.76217
Fold 5: RMSE = 8.77520


7. CVスコアの確認

全foldのOOF予測を使って、最終的なCV RMSEを計算する。
これにより、モデル全体の検証スコアを確認できる。

In [ ]:
# 全foldのOOF予測を使ってCV RMSEを計算
cv_rmse = rmse(y, oof_pred)
print(f"\nCV RMSE: {cv_rmse:.5f}")


CV RMSE: 8.75415


8. 提出ファイルの作成

交差検証で平均したテスト予測を提出形式に整え、CSVファイルとして保存する。

In [ ]:
# 提出ファイルの作成
submission = sample_submission.copy()
submission["exam_score"] = test_pred

# CSVとして保存
submission.to_csv(
    "kaggle_competitions/kaggle_student_test_scores_prediction/output/submission_xgb_baseline.csv",
    index=False
)

print("\nSaved: submission_xgb_baseline.csv")

OSError: Cannot save file into a non-existent directory: 'kaggle_competitions\kaggle_student_test_scores_prediction\output'